# M1-B1 — Modélisation (corrigé)

Tâches 4 à 8. Le pipeline final et les fonctions de découpage sont dans `features.py` et `train.py` :
ce notebook les réutilise pour ne pas dupliquer le code.

In [2]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

engine = create_engine(os.environ["DATABASE_URL"])

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit, cross_validate

import train as T
from model.train_baseline import FEATURES as V1_FEATURES, build_pipeline as build_v1

df = pd.read_sql(text("""
    SELECT h.code AS hotel, r.*
    FROM reservations AS r JOIN hotels AS h ON h.hotel_id = r.hotel_id
    ORDER BY r.arrival_date
"""), engine, parse_dates=["arrival_date"])
train, test = T.split_train_test(df)
y_train, y_test = train["is_canceled"], test["is_canceled"]

## Tâche 4 — Un premier modèle

Toutes les colonnes, encodage sommaire, dates laissées de côté.

In [4]:
X_naif = pd.get_dummies(
    train.drop(columns=["is_canceled", "arrival_date", "reservation_status_date", "reservation_id", "reservation_status"]),
    dtype=int,
).fillna(0)
rf = RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=0)
cross_validate(rf, X_naif, y_train, cv=3, scoring="accuracy")["test_score"]


array([0.84789264, 0.82629994, 0.81563505])

In [5]:
rf.fit(X_naif, y_train)
pd.Series(rf.feature_importances_, X_naif.columns).sort_values(ascending=False).head(5)


lead_time                  0.092436
deposit_type_No Deposit    0.076007
country_PRT                0.075371
previous_cancellations     0.067694
adr                        0.061119
dtype: float64

**Diagnostic.** `reservation_status` est le *dernier* statut de la réservation (`Canceled`, `Check-Out`, `No-Show`) : c'est la réponse elle-même, connue **après** l'événement à prédire. Même raisonnement pour :

- `reservation_status_date` : la date de ce dernier statut ;
- `assigned_room_type` : la chambre attribuée à l'arrivée ;
- `booking_changes` : nombre de modifications, qui peut être mis à jour après la réservation : exclue par prudence ;
- `days_in_waiting_list` et `agent`/`company` sont connus à la réservation : pas de fuite.

Ces colonnes sont exclues de la suite. Dans `train.py`, elles ne sont même pas chargées.

## Tâche 5 — Reproduire la v1 (référence à battre)

## Tâche 6 — Pipeline et variables

Les variables calculées sont dans `features.py` (fonction nommée, importable par l'API).
Comparaison de trois modèles, avec les mêmes variables.

## Tâche 7 — Valider et régler

D'abord avec une validation croisée **aléatoire**, puis **temporelle**

**Choix du modèle.** 

## Tâche 8 — Le verdict

**Coût des erreurs pour l'hôtel :**

- annulation **ratée** (faux négatif) : la chambre reste vide, nuitée perdue. C'est la plainte du service réservations ;
- annulation **prédite à tort** (faux positif) : si l'hôtel survend la chambre, le client arrive et doit être relogé. Coût élevé, mais seulement si l'hôtel agit sur toutes les alertes ; s'il se contente de relancer le client, une fausse alerte coûte peu.

→ **F1** comme métrique principale (équilibre), en suivant le **rappel** de près, puisque c'est le défaut constaté de la v1.

Le seuil de décision est choisi sur les 4 derniers mois de l'**entraînement** (jamais sur le test).

In [ ]:
seuil, trace_seuil = T.choose_threshold(train, best_C)


In [ ]:
v1 = build_v1().
v2_05 = T.build_pipeline(best_C, 0.5).
v2 = T.build_pipeline(best_C, seuil).



In [ ]:
ConfusionMatrixDisplay.from_estimator(v1, test[INPUT_FEATURES], y_test, display_labels=["maintenue", "annulée"]);
ConfusionMatrixDisplay.from_estimator(v2, test[INPUT_FEATURES], y_test, display_labels=["maintenue", "annulée"]);